<div style="background-color:#0e2233;background-image:linear-gradient(135deg,#0b1a29,#123049);color:#ffffff;padding:26px 28px 24px;border-radius:12px;border:1px solid #2a4a66;">
<div style="color:#a0d2ff;font-size:0.78em;font-weight:600;letter-spacing:0.18em;">AI &amp; ROBOTICS · NHÁNH AI ENGINEER · NOTEBOOK THỰC HÀNH</div>
<div style="width:56px;height:4px;background-color:#E07A1F;margin:14px 0 12px;"></div>
<h1 style="color:#ffffff;font-size:2.3em;line-height:1.15;margin:0;border:0;">Chương 47 · LLM hoạt động thế nào</h1>
<p style="color:#cfe6ff;font-size:1.12em;line-height:1.5;margin:10px 0 0;">Mở một LLM thật (Qwen2.5-0.5B) trên CPU, xem phân phối token tiếp theo và perplexity, tự viết vòng sinh văn bản với tham lam, nhiệt độ, top-p, rồi so mô hình gốc với mô hình chỉ dẫn.</p>
</div>

### Sau notebook này bạn sẽ

- Đọc cấu hình một LLM thật và tokenizer của nó.
- Lấy phân phối token tiếp theo, tính perplexity.
- Tự viết vòng sinh văn bản; thấy bịa đặt và lặp vòng.
- Hiểu khuôn hội thoại và khác biệt giữa bản gốc và bản chỉ dẫn.

<div>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Thời gian chạy:</b> khoảng 6 phút trên CPU, chưa kể tải hai mô hình (khoảng 2 GB, một lần)</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Runtime:</b> CPU là đủ, không cần GPU</span>
<span style="display:inline-block;margin:4px 6px 0 0;padding:4px 12px;border-radius:99px;background-color:#FDF1E6;color:#2b2b2b;border:1px solid #E07A1F;font-size:0.9em;"><b>Trang web của chương:</b> <a href="https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/01-llm-la-gi/" style="color:#1F5F8B;">mở bài đọc</a></span>
</div>

### Mục lục

1. [Trang 1 · LLM là gì](#trang-1)
2. [Trang 2 · Đoán token tiếp theo](#trang-2)
3. [Trang 3 · Sinh văn bản](#trang-3)
4. [Trang 4 · Mô hình gốc và mô hình chỉ dẫn](#trang-4)
5. [Trang 5 · Bài tập](#trang-5)

### Cách chạy

Bấm vào một ô code rồi nhấn **Shift + Enter** để chạy ô đó và nhảy sang ô kế tiếp. Muốn chạy hết một lượt thì chọn menu **Runtime > Run all**. Các ô nên chạy theo thứ tự từ trên xuống, vì ô sau dùng biến của ô trước.

<a name="trang-1" id="trang-1"></a>

## Trang 1 · LLM là gì

Tải Qwen2.5-0.5B (khoảng 1 GB), đọc cấu hình, đo tokenizer trên UIT-VSFC.

*Đọc kèm trên web: [Trang 1 · LLM là gì](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/01-llm-la-gi/)*

In [ ]:
!pip install -q pyarrow transformers                         # đọc parquet, mô hình Hugging Face

In [ ]:
import math                                                 # hàm mũ
import os                                                   # đường dẫn
import time                                                 # đo thời gian
import urllib.request                                       # tải tệp
import numpy as np                                          # mảng
import pandas as pd                                         # bảng
import torch                                                # PyTorch
from transformers import AutoTokenizer, AutoModelForCausalLM   # tokenizer, mô hình đoán token tiếp theo
torch.set_num_threads(4)                                    # 4 luồng
R_GOC = "060db6499f32faf8b98477b0a26969ef7d8b9987"          # ghim phiên bản
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B", revision=R_GOC)   # tokenizer
llm = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B", revision=R_GOC, dtype=torch.float32).eval()   # bản gốc
c = llm.config                                              # cấu hình
print(sum(p.numel() for p in llm.parameters()), len(tok))   # số tham số, số token
print(c.num_hidden_layers, c.hidden_size, c.num_attention_heads, c.num_key_value_heads,
      c.intermediate_size, c.max_position_embeddings, c.vocab_size)   # khối, d, đầu query, đầu key-value, lớp ẩn, ngữ cảnh, từ điển

Output mong đợi:

```text
494032768 151665
24 896 14 2 4864 32768 151936
```

In [ ]:
URL = "https://huggingface.co/api/datasets/uitnlp/vietnamese_students_feedback/parquet/default/{}/0.parquet"   # UIT-VSFC
THU_MUC = os.path.expanduser("~/.cache/vsfc")               # lưu tạm
def tai(phan):                                              # tải một lần
    os.makedirs(THU_MUC, exist_ok=True)                     # tạo thư mục
    tep = os.path.join(THU_MUC, phan + ".parquet")          # đường dẫn
    if not os.path.exists(tep):                             # chưa có thì tải
        urllib.request.urlretrieve(URL.format(phan), tep)   # tải về
    return pd.read_parquet(tep)                             # đọc
va = tai("validation")                                      # val
print(round(np.mean([len(tok.tokenize(s)) for s in va.sentence]), 2))   # token trung bình mỗi câu
cau = "Thầy giảng dạy rất nhiệt tình, slide đầy đủ."         # câu thử của Chương 45
print(len(tok.tokenize(cau)), [tok.convert_tokens_to_string([t]) for t in tok.tokenize(cau)])   # số token, từng token

Output mong đợi:

```text
15.51
12 ['Th', 'ầy', ' giảng', ' dạy', ' rất', ' nhiệt', ' tình', ',', ' slide', ' đầy', ' đủ', '.']
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">15,51 token mỗi câu, so với 49,77 của GPT-2 ở Chương 45: gần như mỗi âm tiết một token.</p>
</div>

<a name="trang-2" id="trang-2"></a>

## Trang 2 · Đoán token tiếp theo

Phân phối token tiếp theo và perplexity (vài giây).

*Đọc kèm trên web: [Trang 2 · Đoán token tiếp theo](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/02-doan-token-tiep-theo/)*

In [ ]:
def tiep_theo(van_ban, k=8):                                # k token tiếp theo có xác suất cao nhất
    x = tok(van_ban, return_tensors="pt")                   # mã hóa
    with torch.no_grad():                                   # không cần gradient
        p = llm(**x).logits[0, -1].softmax(-1)              # phân phối cho token ngay sau văn bản
    xs, ids = p.topk(k)                                     # k token tốt nhất
    return p, [(tok.decode(i), round(v.item(), 3)) for v, i in zip(xs, ids)]   # phân phối, danh sách
for van_ban in ["Thủ đô của Việt Nam là", "Hôm nay trời mưa nên em"]:   # hai đoạn mở đầu
    print(van_ban, tiep_theo(van_ban)[1])                   # top 8
p, _ = tiep_theo("Thủ đô của Việt Nam là")                  # phân phối đầy đủ
for w in [" Hà", " thành"]:                                 # hai token đáng xem
    i = tok.encode(w)[0]                                    # chỉ số token
    print(repr(w), round(p[i].item(), 4), int((p > p[i]).sum()) + 1)   # xác suất và thứ hạng

Output mong đợi:

```text
Thủ đô của Việt Nam là [(' một', 0.136), (' nơi', 0.13), (' gì', 0.115), (' thành', 0.108), (' thủ', 0.09), (' __', 0.02), (' địa', 0.017), (' đất', 0.013)]
Hôm nay trời mưa nên em [(' đi', 0.144), (' không', 0.078), (' đã', 0.036), (' m', 0.025), (' về', 0.019), (' sẽ', 0.018), (' phải', 0.017), (' lại', 0.016)]
' Hà' 0.0062 14
' thành' 0.108 4
```

In [ ]:
import random                                               # xáo trộn
def perplexity(van_ban):                                    # e mũ cross-entropy trung bình mỗi token
    ids = tok(van_ban, return_tensors="pt")["input_ids"]    # mã hóa
    with torch.no_grad():                                   # không cần gradient
        return round(math.exp(llm(ids, labels=ids).loss.item()), 1)   # mô hình tự dịch nhãn một vị trí
cau = "Thầy giảng dạy rất nhiệt tình và luôn giúp đỡ sinh viên."   # câu tự nhiên
tu = cau[:-1].split()                                       # các từ
random.seed(0)                                              # xáo cố định
random.shuffle(tu)                                          # xáo trộn
xao = " ".join(tu) + "."                                    # câu xáo
print(perplexity(cau), cau)                                 # câu tự nhiên
print(perplexity(xao), xao)                                 # câu xáo

Output mong đợi:

```text
31.6 Thầy giảng dạy rất nhiệt tình và luôn giúp đỡ sinh viên.
3654.4 giảng đỡ giúp tình sinh dạy rất luôn nhiệt Thầy viên và.
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Cùng các từ, xáo trộn thì perplexity tăng khoảng 115 lần.</p>
</div>

<a name="trang-3" id="trang-3"></a>

## Trang 3 · Sinh văn bản

Vòng sinh văn bản tự viết: tham lam, nhiệt độ, top-p (khoảng nửa phút).

*Đọc kèm trên web: [Trang 3 · Sinh văn bản](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/03-sinh-van-ban/)*

In [ ]:
def sinh(van_ban, so_token=30, T=0.0, top_p=1.0, seed=0):   # sinh so_token token
    g = torch.Generator().manual_seed(seed)                 # ngẫu nhiên cố định
    ids = tok(van_ban, return_tensors="pt")["input_ids"]    # đoạn mở đầu
    for _ in range(so_token):                               # mỗi bước một token
        with torch.no_grad():                               # không cần gradient
            lg = llm(ids).logits[0, -1]                     # chỉ cần vị trí cuối
        if T == 0:                                          # tham lam
            j = lg.argmax()                                 # token xác suất cao nhất
        else:                                               # lấy mẫu
            p = (lg / T).softmax(-1)                        # nhiệt độ
            if top_p < 1:                                   # top-p
                xs, thu_tu = p.sort(descending=True)        # sắp giảm dần
                giu = xs.cumsum(0) - xs < top_p             # giữ token cho tới khi tổng vừa vượt top_p
                p = torch.zeros_like(p).scatter(0, thu_tu[giu], xs[giu])   # bỏ cái đuôi
                p = p / p.sum()                             # chuẩn hóa lại
            j = torch.multinomial(p, 1, generator=g)[0]     # rút một token
        ids = torch.cat([ids, j.view(1, 1)], 1)             # nối rồi đoán tiếp
    return tok.decode(ids[0])                               # văn bản
t0 = time.time()                                            # đo thời gian
print(repr(sinh("Thủ đô của Việt Nam là thành phố")))       # tham lam
print(repr(sinh("Trường đại học tốt nhất để học trí tuệ nhân tạo là")))   # tham lam
print(time.time() - t0 < 120)                               # dưới 2 phút

Output mong đợi:

```text
'Thủ đô của Việt Nam là thành phố lớn nhất thế giới, với hơn 200 triệu dân, 100 triệu người dân ở đây. Trong đó, có '
'Trường đại học tốt nhất để học trí tuệ nhân tạo là gì? - Tin tức - Trường đại học tốt nhất để học trí tuệ nhân tạo là gì?\nTrường đại học tốt nhất để học trí'
True
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Câu đầu trôi chảy và bịa hoàn toàn; câu sau lặp vòng.</p>
</div>

In [ ]:
for T in (0.7, 1.0, 1.5):                                   # ba nhiệt độ
    print(T, repr(sinh("Hôm nay trời mưa nên em", T=T)))    # cùng seed
print("top-p 0.9", repr(sinh("Hôm nay trời mưa nên em", T=1.0, top_p=0.9)))   # cắt đuôi

Output mong đợi:

```text
0.7 'Hôm nay trời mưa nên em rất thích đi bơi. Thấy em làm vậy em cảm thấy rất tự hào, vì em đã tự mình làm điều này, mình cũng'
1.0 'Hôm nay trời mưa nên em họ tập không bầm bầm ở tối. Yêu Em hiện tại em Thiếu Ngọc mới bị kiểu kiểu chuyện như thế này, sao'
1.5 'Hôm nay trời mưa nên em họ ta không憩 mùa愈旺 ởpellier啦 //AsVip一期 Aw lined da chorus perché ultimo contest headline serfs Fo_literals each Ausb'
top-p 0.9 'Hôm nay trời mưa nên em họ tập không bọc luôn có ở thì làm gì em yêu tôi thấy em ho một đôi mắt bị mèo che thấy đâu nè\n'
```

<a name="trang-4" id="trang-4"></a>

## Trang 4 · Mô hình gốc và mô hình chỉ dẫn

Bản Instruct (tải thêm khoảng 1 GB), khuôn hội thoại, hỏi cả hai bản.

*Đọc kèm trên web: [Trang 4 · Mô hình gốc và mô hình chỉ dẫn](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/04-goc-va-instruct/)*

In [ ]:
R_CD = "7ae557604adf67be50417f59c2c2f167def9a775"           # ghim phiên bản
tok_cd = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", revision=R_CD)   # tokenizer kèm khuôn hội thoại
llm_cd = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", revision=R_CD, dtype=torch.float32).eval()   # bản chỉ dẫn
hoi = "Thủ đô của Việt Nam là gì?"                         # câu hỏi
khuon = tok_cd.apply_chat_template([{"role": "user", "content": hoi}], tokenize=False, add_generation_prompt=True)   # bọc trong khuôn
print(khuon)                                                # xem khuôn

Output mong đợi:

```text
<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
Thủ đô của Việt Nam là gì?<|im_end|>
<|im_start|>assistant
```

In [ ]:
HET = tok_cd.convert_tokens_to_ids("<|im_end|>")           # token kết thúc lượt nói
def tra_loi(mo_hinh, van_ban, so_token=40):                 # sinh tham lam, dừng ở token kết thúc
    ids = tok_cd(van_ban, return_tensors="pt")["input_ids"] # mã hóa
    n0 = ids.shape[1]                                       # độ dài đoạn đầu
    for _ in range(so_token):                               # mỗi bước
        with torch.no_grad():                               # không cần gradient
            j = mo_hinh(ids).logits[0, -1].argmax()         # tham lam
        ids = torch.cat([ids, j.view(1, 1)], 1)             # nối
        if j.item() in (HET, tok_cd.eos_token_id):          # mô hình tự kết thúc lượt nói
            break                                           # dừng
    return tok_cd.decode(ids[0, n0:])                       # chỉ phần mô hình viết
print("gốc, hỏi thẳng:   ", repr(tra_loi(llm, hoi)))        # bản gốc, không khuôn
print("chỉ dẫn, có khuôn:", repr(tra_loi(llm_cd, khuon)))   # bản chỉ dẫn, có khuôn
print("gốc, có khuôn:    ", repr(tra_loi(llm, khuon)))      # bản gốc, có khuôn

Output mong đợi:

```text
gốc, hỏi thẳng:    ' - Tin tức - 24h.vn\nThủ đô của Việt Nam là gì?\nThủ đô của Việt Nam là gì? 24h.vn\nThủ đô của Việt Nam'
chỉ dẫn, có khuôn: 'Thủ đô của Việt Nam là Hà Nội.<|im_end|>'
gốc, có khuôn:     'Thủ đô của Việt Nam là Hà Nội.acco\nacco\nBạn có thể sử dụng các công cụ để tìm kiếm thông tin về thủ đô của Việt Nam.acco\nacco\nBạn có'
```

<div style="border-left:6px solid #2E8B57;background-color:#EAF6EF;color:#2b2b2b;padding:12px 18px 14px;border-radius:8px;margin:10px 0;">
<div style="color:#1E6B42;font-weight:700;font-size:0.85em;letter-spacing:0.08em;">ĐỌC KẾT QUẢ</div>
<p style="margin:8px 0 0;color:#2b2b2b;line-height:1.6;">Bản gốc có khuôn cũng trả lời đúng Hà Nội nhưng không biết dừng: kiến thức đến từ tiền huấn luyện, bước chỉ dẫn dạy hình thức.</p>
</div>

<a name="trang-5" id="trang-5"></a>

## Trang 5 · Bài tập

Ba bài tập. Đoán trước, rồi mới mở đáp án.

*Đọc kèm trên web: [Trang 5 · Bài tập](https://ai-and-robotics-handbook.vercel.app/ai-engineer/ch47-llm-hoat-dong/05-bai-tap/)*

### Bài 47.1 · Đếm tham số bằng tay

Từ cấu hình, đếm số tham số của Qwen2.5-0.5B và phần trăm của bảng vector token.
<details><summary>Xem đáp án</summary>

Mỗi khối 14 912 384, tổng khớp đúng 494 032 768; bảng vector token chiếm 27,6%. Code ở ô ngay dưới.
</details>

In [ ]:
d, f, V, L, dk = 896, 4864, 151936, 24, 64                  # cấu hình
kv = 2 * dk                                                 # 2 đầu key, value dùng chung cho 14 đầu query
moi_khoi = (d * d + d) + 2 * (d * kv + kv) + d * d + 3 * d * f + 2 * d   # query, key và value, đầu ra, SwiGLU, hai RMSNorm
tong = L * moi_khoi + V * d + d                             # + bảng vector token + RMSNorm cuối
print(moi_khoi, tong, tong == sum(p.numel() for p in llm.parameters()), round(100 * V * d / tong, 1))   # khớp không, % bảng token

Output mong đợi:

```text
14912384 494032768 True 27.6
```

### Bài 47.2 · Phân loại cảm xúc không cần huấn luyện

So log xác suất của Tích cực và Tiêu cực; ngưỡng 0, rồi ngưỡng chọn trên 100 câu có nhãn.
<details><summary>Xem đáp án</summary>

Ngưỡng 0: 52,5%, mô hình lệch hẳn về Tiêu cực. Ngưỡng chọn trên 100 câu khác: 81%. Code ở ô ngay dưới (khoảng 3 phút).
</details>

In [ ]:
def log_xs(khuon, tra_loi):                                 # tổng log xác suất của tra_loi khi đứng sau khuon
    a = tok_cd(khuon, return_tensors="pt")["input_ids"]     # câu lệnh
    b = tok_cd(tra_loi, return_tensors="pt")["input_ids"]   # câu trả lời
    ids = torch.cat([a, b], 1)                              # nối
    with torch.no_grad():                                   # không cần gradient
        lp = llm_cd(ids).logits[0, :-1].log_softmax(-1)     # log xác suất ở mọi vị trí
    vt = torch.arange(a.shape[1] - 1, ids.shape[1] - 1)     # vị trí đoán từng token của tra_loi
    return lp[vt, ids[0, a.shape[1]:]].sum().item()         # cộng lại
def diem(s):                                                # điểm tích cực trừ tiêu cực
    k = tok_cd.apply_chat_template([{"role": "user", "content": f'Cảm xúc của câu sau là tích cực hay tiêu cực? "{s}"'}],
                                   tokenize=False, add_generation_prompt=True)   # câu lệnh trong khuôn
    return log_xs(k, "Tích cực") - log_xs(k, "Tiêu cực")    # hiệu log xác suất
hai_lop = va[va.sentiment != 1]                             # bỏ trung tính
chon, do = hai_lop.iloc[:100], hai_lop.iloc[100:300]        # 100 câu để chọn ngưỡng, 200 câu để đo
d_chon = np.array([diem(s) for s in chon.sentence])         # điểm 100 câu chọn
d_do = np.array([diem(s) for s in do.sentence])             # điểm 200 câu đo
y_chon, y_do = chon.sentiment.values == 2, do.sentiment.values == 2   # nhãn tích cực
print(np.bincount(do.sentiment, minlength=3), (d_do > 0).sum(), ((d_do > 0) == y_do).mean())   # phân bố nhãn, số câu đoán tích cực, độ chính xác
nguong = max(sorted(d_chon), key=lambda t: ((d_chon > t) == y_chon).mean())   # ngưỡng tốt nhất trên 100 câu chọn
print(round(nguong, 2), ((d_do > nguong) == y_do).mean())   # ngưỡng, độ chính xác trên 200 câu đo

Output mong đợi:

```text
[ 90   0 110] 17 0.525
-0.82 0.81
```

### Bài 47.3 · Vì sao tham lam hay lặp

Tỷ lệ bộ ba token khác nhau trong 100 token sinh bằng tham lam và bằng top-p 0,9, nhiệt độ 0,7.
<details><summary>Xem đáp án</summary>

27,6% so với 79,6%: tham lam lặp gần ba phần tư văn bản. Code ở ô ngay dưới (khoảng 1 phút).
</details>

In [ ]:
def ty_le_khac(van_ban, mo_dau):                            # tỷ lệ bộ ba token khác nhau
    ids = tok(van_ban)["input_ids"][len(tok(mo_dau)["input_ids"]):]   # chỉ phần được sinh
    ba = list(zip(ids, ids[1:], ids[2:]))                   # các bộ ba
    return round(len(set(ba)) / len(ba), 3)                 # khác nhau trên tổng
md = "Trường đại học tốt nhất để học trí tuệ nhân tạo là"   # đoạn mở đầu
print("tham lam", ty_le_khac(sinh(md, so_token=100), md))   # tham lam
print("top-p 0.9, T 0.7", ty_le_khac(sinh(md, so_token=100, T=0.7, top_p=0.9), md))   # lấy mẫu

Output mong đợi:

```text
tham lam 0.276
top-p 0.9, T 0.7 0.796
```